In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]  = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"]  = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]  = os.getenv("GROQ_API_KEY")

In [3]:
from langchain.chat_models import init_chat_model

# model = init_chat_model("gemini-2.5-flash",model_provider="google_genai")

model = init_chat_model("google_genai:gemini-2.5-flash-lite")
model

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x1244997f0>, default_metadata=(), model_kwargs={})

In [ ]:
##Invoke model

response=model.invoke("hello how are you?")
response

In [ ]:
response.content

In [4]:
import os 
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3-32b")
response = model.invoke("why do paret talk in 15 words")
response

AIMessage(content='<think>\nOkay, the user is asking why "paret talk in 15 words." Wait, "paret" might be a typo. Maybe they meant "parent" or "parent talk"? Let me check the context. The user wrote "why do paret talk in 15 words." Hmm, "paret" doesn\'t make sense. Let me think. Could it be a misspelling of "parent"? If so, the question would be about why parents talk in 15 words. Alternatively, maybe "parental talk"? But that doesn\'t fit either. Another possibility is "parent talk in 15 words" as in concise communication. Let me consider the possible corrections.\n\nIf the user intended "parent talk in 15 words," maybe they\'re asking about why parents use short sentences or concise communication with children. Parents often use short, clear sentences to help children understand better. So the answer would revolve around simplicity, clarity, and child development. Alternatively, if "paret" is a specific term or name, maybe it\'s a typo for "parent" or "parent talk." Let me verify if 

In [ ]:
## Streaming and Batch

## Streaming and Batch

###Streaming - where we can start showing o/p content the movement it gets generated 

###Batch is to generate more then one parallel responses (model.batch(["<questions1>","<questions2>","<questions3>"]))

In [5]:
from langchain.chat_models import init_chat_model

# model = init_chat_model("gemini-2.5-flash",model_provider="google_genai")

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [ ]:
for chunk in model.stream("write me a 500 words paragraph on AI Cyber security "):
    print(chunk.text, flush=True)


In [ ]:
model.batch([])

# Tools
it can be any independantely like , browser , the one which will use to extract data OR use the Api of any platform the gather real time info, like LLMs are not trained on real time data so we use tools to get correct info, like wather



In [6]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""      #DOC string 
    return f"if sunny in {location}"



model_with_tools = model.bind_tools([get_weather])


In [17]:
from langchain_core.messages import content
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage

import os 
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3-32b")
# response = model.invoke("why do paret talk in 15 words")

# model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
model_with_tools = model.bind_tools([get_weather])
# messages = [{"role":"user","content":"what is the weather like in pune?"}]
messages = [HumanMessage(content="what is the weather like in pune?")]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

print(messages)
# print(response.text)

# for tool_call in response.tool_calls:

#     tool_result = get_weather.invoke(tool_call["args"])
#     messages.append(tool_call)
#     # print(f"Tool: {tool_call['name']}")
#     # print(f"Args: {tool_call['args']}")

for tc in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tc["args"])
    messages.append(
        ToolMessage(
            content=str(tool_result),
            tool_call_id=tc["id"],
            name=tc["name"],
        )
    )

final_resoponse = model_with_tools.invoke(messages)
# print(final_resoponse.content)

# for chunk in model.stream(final_resoponse):
#     print(chunk.text, flush=True)



[HumanMessage(content='what is the weather like in pune?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Pune. I need to use the get_weather function. The function requires the location parameter. Pune is a city in India, so I should specify "Pune, India" as the location. Let me check if there\'s any other required parameters. No, just location. So I\'ll call the function with {"location": "Pune, India"}.\n', 'tool_calls': [{'id': 'xw6bqgqsg', 'function': {'arguments': '{"location":"Pune, India"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 155, 'total_tokens': 264, 'completion_time': 0.1769973, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.013706411, 'prompt_tokens_details': None, 'queue_time': 0.051498609, 'total_time': 0.190703711}, 'model_name': 'qwen/qwen3-32b', 'syste

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage

model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage(content="what is the temperature in pune?")]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tc in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tc["args"])
    messages.append(
        ToolMessage(
            content=str(tool_result),
            tool_call_id=tc["id"],
            name=tc["name"],
        )
    )
 
final_response = model_with_tools.invoke(messages)
print(final_response.content)